# ForestWatch Papua — Improve Model (Fine-tuning Kelas Minoritas)

**Tujuan**: naikkan IoU 5 kelas lemah (Lahan Terbuka, Sawit, Pertanian Lain, Tambang,
Permukiman) **tanpa menurunkan** 2 kelas mayoritas (Perairan, Hutan), di atas checkpoint
**model_1 (Attention U-Net)** yang sudah dilatih — **tanpa train ulang dari nol**.

**Dataset**: `Bahan_Training_Fix_Combined_v3` (lihat `export_train_only_gel6.ipynb` bagian T8) —
Papua lama + gelombang-4 (train/val/test, eval-only, split per-region aman dari leakage) +
gelombang-5 + gelombang-6 (train-only, ~8.076 patch tambahan utk 5 kelas lemah). val/test FROZEN
sejak gelombang-4 (jadi acuan jujur, tidak ikut membesar di gelombang-5/6).

**Metode (decoupled fine-tuning — Kang dkk. ICLR 2020, arXiv 1910.09217):**
- Encoder ResNet50 **DIBEKUKAN** (fitur ImageNet paling generalizable) → latih ulang
  **decoder + segmentation head** saja (cukup utk perbaiki separabilitas kelas yang sering
  tertukar: Tambang↔Lahan Terbuka, Sawit↔Pertanian Lain).
- Loss `0.4·Focal + 0.4·Tversky(β>α) + 0.2·CE` — **SEMUA unweighted** (Tversky β=0,7>α=0,3
  menekan false-negative → recall minoritas naik, Abraham & Khan 2019). Rebalance kelas
  SENGAJA hanya lewat `WeightedRandomSampler` class-balanced murni (Kang dkk. 2020, cRT) —
  bukan ditumpuk dgn median-frequency class-weight di loss (Eigen & Fergus 2015, dipakai di
  `train_model_1_attention_unet.ipynb` utk training awal) — biar efek sampler kelihatan jelas,
  tidak campur 2 mekanisme rebalance sekaligus.
- **Seleksi checkpoint terjaga (guarded)**: tiap epoch, model hanya disimpan bila IoU kelas
  mayoritas (Perairan, Hutan) di **VAL** tidak turun > ε dari baseline → model final dijamin
  tak mengorbankan kelas mayoritas.
- **Anti-bocor**: seleksi pakai VAL; TEST hanya sekali di akhir untuk laporan jujur.
- **Reversible**: hasil → `best_model_finetune.pt` terpisah; `best_model.pt` baseline tak disentuh.

**Cara pakai**: jalankan cell berurutan dari atas. Cell fine-tune aman di-resume bila sesi mati.

**Referensi**: Kang dkk. ICLR 2020 (decoupling/cRT); Oktay dkk. MICCAI 2018, arXiv 1804.03999
(attention-gate, dasar pilih model_1); Abraham & Khan 2019 (focal-Tversky); Eigen & Fergus 2015
(median-frequency — dipakai di training awal, TIDAK di notebook fine-tune ini, lihat penjelasan
di atas).


## Bagian 0 — Setup environment (Colab / Lab / Kaggle)


In [ ]:
# === Bagian 0 — Setup (set ENV = "colab" / "lab" / "kaggle") ===
ENV = "colab"   # "colab" | "lab" (PC+Drive Desktop) | "kaggle"
import os, sys, subprocess, importlib
from pathlib import Path

# Auto-deteksi Kaggle (folder /kaggle hanya ada di runtime Kaggle).
if Path("/kaggle").exists() and ENV != "kaggle":
    print(f"[auto-detect] /kaggle -> override ENV='{ENV}' -> 'kaggle'")
    ENV = "kaggle"

DRIVE_ROOT = None
_SRC = None
if ENV == "colab":
    subprocess.run("cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
                   "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
                   shell=True, check=False)
    subprocess.run("pip install -q -e /content/fw_repo[ml]", shell=True, check=False)
    _SRC = "/content/fw_repo/model/src"
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
elif ENV == "lab":
    DRIVE_ROOT = Path(r"G:/My Drive/Satria Data 3.0")   # <- SESUAIKAN mount Drive Desktop lab
elif ENV == "kaggle":
    subprocess.run("cd /kaggle/working && (git -C fw_repo pull -q || git clone --depth 1 "
                   "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
                   shell=True, check=False)
    subprocess.run("pip install -q --no-deps -e /kaggle/working/fw_repo[ml]", shell=True, check=False)
    subprocess.run("pip install -q --no-deps segmentation-models-pytorch albumentations torchmetrics",
                   shell=True, check=False)
    _SRC = "/kaggle/working/fw_repo/model/src"
else:
    raise ValueError("ENV harus 'colab' | 'lab' | 'kaggle'")

if _SRC and _SRC not in sys.path:
    sys.path.insert(0, _SRC)
for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

import torch
_gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"ENV={ENV} | CUDA={torch.cuda.is_available()}{_gpu}")


In [ ]:
# === Deklarasi path, identitas model, & config ===
from forestwatch.config import load_config
from forestwatch.utils.io import save_json, load_json
from forestwatch.constants import N_CLASSES, CLASS_NAMES, CLASS_COLORS
cfg = load_config()

MODEL_KEY  = "model_1_attention_unet"   # pemenang banding (compare_and_select_best_model.ipynb)
MODEL_ARCH = dict(architecture="unet_scse", encoder_name="resnet50")
STRONG = [0, 1]            # Perairan, Hutan -- JAGA (jangan turun)
WEAK   = [2, 3, 4, 5, 6]   # Lahan Terbuka, Sawit, Pertanian Lain, Tambang, Permukiman -- target naik

# Folder BARU khusus hasil fine-tune (terpisah dari Model_Comparison/<model_key> yg isinya
# 3 model baseline) -- sejajar (sibling) di ForestWatch_Outputs, gampang dicari.
# DRIVE_TARGET_NAME dipakai juga di cell promosi (kaggle) sbg nama folder tujuan unggah manual.
DRIVE_TARGET_NAME = f"ForestWatch_Outputs/Model_Improve/{MODEL_KEY}"

if ENV in ("colab", "lab"):
    # v3 (lihat export_train_only_gel6.ipynb bagian T8) = Papua lama + gel.4 (train/val/test,
    # split per-region aman dari leakage) + gel.5 + gel.6 (train-only, 5 kelas lemah) sudah
    # DIGABUNG & DIPAKET di sana -- jangan pakai "Bahan_Training_Fix" (Papua-only) lagi di sini,
    # train_new gel.5/gel.6 (~8.076 patch) tidak akan kepakai kalau masih nunjuk ke yang lama.
    BAHAN_DIR   = DRIVE_ROOT / "Bahan_Training_Fix_Combined_v3"
    MODELS_ROOT = DRIVE_ROOT / "ForestWatch_Outputs" / "Model_Comparison"   # baseline 3 model (READ-ONLY di sini)
    BASE_DIR    = MODELS_ROOT / MODEL_KEY
    BASE_CKPT   = BASE_DIR / "best_model.pt"
    FT_DIR      = DRIVE_ROOT / "ForestWatch_Outputs" / "Model_Improve" / MODEL_KEY   # folder BARU
else:  # kaggle: data + checkpoint baseline dari dataset yg di-attach (Add Input)
    # Attach dataset Bahan_Training_Fix_Combined_v3 (bukan v1/v2 yg lama) sbg Kaggle Input.
    BAHAN_DIR = None                       # diisi di cell ekstrak (rglob)
    BASE_DIR  = None
    BASE_CKPT = next(Path("/kaggle/input").rglob("best_model.pt"))
    FT_DIR    = Path("/kaggle/working") / "Model_Improve" / MODEL_KEY   # lokal sesi; lihat cell promosi utk unduh

FT_DIR.mkdir(parents=True, exist_ok=True)
FT_CKPT    = FT_DIR / "best_model_finetune.pt"
FT_RESUME  = FT_DIR / "best_model_finetune_resume.pt"
FT_SAMPLER = FT_DIR / "class_presence_finetune.json"   # cache VEKTOR kehadiran kelas (class-balanced sampling, BUKAN skalar lama)
OUT_DIR    = FT_DIR

assert BASE_CKPT.exists(), f"Baseline checkpoint tak ada: {BASE_CKPT}"
print("Baseline ckpt :", BASE_CKPT)
print("Dataset       :", BAHAN_DIR if BAHAN_DIR else "(kaggle, attach Bahan_Training_Fix_Combined_v3)")
print("Output FT dir :", FT_DIR)
print("Kelas JAGA    :", [CLASS_NAMES[c] for c in STRONG])
print("Kelas target  :", [CLASS_NAMES[c] for c in WEAK])


In [ ]:
# === Ekstrak dataset ke disk lokal + daftar file (train/val/test) ===
# Pola sama spt notebook training: extract .tar -> disk lokal (lepas dari bottleneck Drive FUSE).
# BAHAN_DIR sudah menunjuk ke Bahan_Training_Fix_Combined_v3 (cell sebelumnya) -- train/val/test
# di sini SUDAH gabungan Papua lama + gel.4 (train/val/test) + gel.5 + gel.6 (train-only).
# Tidak perlu merge manual lagi spt revisi sebelumnya (cell itu sudah dihapus -- akan
# double-count gel.4 val/test kalau dijalankan lagi di atas v3).
from forestwatch.data.dataset import extract_dataset_archives
from forestwatch.data.patches import list_patches

if ENV in ("colab", "lab"):
    LOCAL_DIR = Path("/content/dataset_local") if ENV == "colab" else Path.home() / "dataset_local"
    _splits = ("train", "val", "test")
    if (BAHAN_DIR / "train_rajaampat").exists():
        _splits = _splits + ("train_rajaampat",)
    local_dirs = extract_dataset_archives(BAHAN_DIR, LOCAL_DIR, splits=_splits, max_workers=8)
    cw_path = BAHAN_DIR / "class_weights.json"
else:  # kaggle
    BAHAN_SRC = Path("/kaggle/temp/bahan_src"); BAHAN_SRC.mkdir(parents=True, exist_ok=True)
    # Cari file "<split>_part*.tar" LANGSUNG lewat nama file, di MANA PUN di bawah /kaggle/input
    # -- bukan cari folder literal bernama "train"/"val"/"test". Sengaja begini (bukan symlink
    # folder spt revisi sebelumnya) karena:
    # (a) tahan lepas dari struktur folder apapun hasil Kaggle CLI saat upload (flat atau
    #     dibungkus --dir-mode tar/zip -- keduanya ternyata rawan: folder ke-skip diam-diam
    #     kalau tanpa --dir-mode, atau malah jadi 1 file wrapper "<split>.tar" kalau pakai
    #     --dir-mode, BUKAN folder berisi part-part aslinya).
    # (b) otomatis MENGGABUNG train kalau dipecah jadi >1 Kaggle Dataset (train-1, train-2, dst,
    #     krn ukurannya besar) -- tinggal cocokkan semua file by nama, lepas dari dataset mana.
    for split in ("train", "val", "test", "train_rajaampat"):
        (BAHAN_SRC / split).mkdir(parents=True, exist_ok=True)
        _found = list(Path("/kaggle/input").rglob(f"{split}_part*.tar"))
        for _tf in _found:
            _dst = BAHAN_SRC / split / _tf.name
            if not _dst.exists():
                os.symlink(_tf, _dst)
        if _found:
            print(f"{split}: {len(_found)} file .tar ditemukan -> {BAHAN_SRC / split}")

    _f = list(Path("/kaggle/input").rglob("class_weights.json"))
    if _f and not (BAHAN_SRC / "class_weights.json").exists():
        os.symlink(_f[0], BAHAN_SRC / "class_weights.json")

    _splits = tuple(s for s in ("train", "val", "test", "train_rajaampat")
                     if next((BAHAN_SRC / s).glob("*"), None) is not None)
    local_dirs = extract_dataset_archives(BAHAN_SRC, Path("/kaggle/temp/dataset_local"),
                                           splits=_splits, max_workers=8)
    cw_path = BAHAN_SRC / "class_weights.json"

final_train_files = list_patches(local_dirs["train"])
if "train_rajaampat" in local_dirs:
    _ra = list_patches(local_dirs["train_rajaampat"])
    final_train_files = final_train_files + _ra
    print(f"  + {len(_ra)} patch Raja Ampat (Tambang asli)")
val_p  = list_patches(local_dirs["val"])
test_p = list_patches(local_dirs["test"])
assert final_train_files and val_p and test_p, "train/val/test kosong -- cek BAHAN_DIR / attach dataset."
print(f"train={len(final_train_files)} | val={len(val_p)} | test={len(test_p)}")


In [ ]:
# === Sampler: class-balanced MURNI (Kang dkk. 2020, cRT) -- BUKAN reweighting frekuensi ===
# Beda dari train_model_*.ipynb (compute_patch_sampler_weights -- proporsional ke
# kelangkaan piksel: kelas langka lebih SERING tapi tak SAMA): di sini tiap kelas (Hutan
# maupun Tambang) dapat probabilitas TERPILIH SAMA (1/7) per step, lepas dari piksel
# aslinya -- resep literal cRT stage classifier-retraining.
import json
import numpy as np
from forestwatch.data.dataset import compute_class_balanced_sampler_weights

train_sampler_weights = compute_class_balanced_sampler_weights(
    final_train_files, n_classes=N_CLASSES, cache_path=FT_SAMPLER,
)

# Diagnostik: N_c (jumlah patch yang mengandung kelas c) -- makin kecil N_c, makin besar
# bobot per-patch yang didapat kelas itu (itu yang bikin probabilitas terpilih jadi SAMA).
_presence = np.array(list(json.load(open(FT_SAMPLER)).values()))
n_c = _presence.sum(axis=0)
print(f"{'kelas':<16}{'N_c (patch)':>13}{'bobot/patch':>13}")
for c in range(N_CLASSES):
    w_c = (1 / N_CLASSES) / max(n_c[c], 1)
    print(f"{CLASS_NAMES[c]:<16}{int(n_c[c]):>13,}{w_c:>13.6f}")


In [ ]:
# === Build DataLoaders (train: sampler class-balanced MANUAL) + model (encoder BEKU) + loss ===
from forestwatch.data.dataset import PapuaDataset
from forestwatch.model.architecture import build_unet, count_parameters
from forestwatch.model.losses import make_loss_fn
from forestwatch.training.trainer import set_encoder_trainable
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch

N_WORKERS = 2 if ENV == "colab" else min(8, (os.cpu_count() or 2))

# Train loader dibangun MANUAL (bukan build_dataloaders_from_files) -- fungsi itu pakai
# compute_patch_sampler_weights (proporsional frekuensi); kita pakai bobot
# class-balanced dari cell sebelumnya (skema beda total, bukan variasi parameter saja).
train_ds = PapuaDataset(final_train_files, train=True, augment_p=cfg["training"]["augmentation"])
sampler = WeightedRandomSampler(
    weights=train_sampler_weights, num_samples=len(final_train_files), replacement=True,
)
train_loader = DataLoader(
    train_ds, batch_size=cfg["training"]["batch_size"], sampler=sampler,
    num_workers=N_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=(N_WORKERS > 0),
)
val_loader = DataLoader(
    PapuaDataset(val_p, train=False), batch_size=cfg["training"]["batch_size"],
    shuffle=False, num_workers=N_WORKERS, pin_memory=True,
    persistent_workers=(N_WORKERS > 0),
)
print(f"Train {len(train_loader.dataset)} | Val {len(val_loader.dataset)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_unet(in_channels=cfg["model"]["in_channels"], classes=cfg["model"]["classes"],
                   encoder_weights=cfg["model"]["encoder_weights"], **MODEL_ARCH)
model.load_state_dict(torch.load(BASE_CKPT, map_location="cpu"))
model = model.to(device)

# DECOUPLED FINE-TUNE: bekukan encoder (fitur ImageNet generalizable, Kang dkk. 2020).
# decoder + segmentation_head tetap trainable.
assert set_encoder_trainable(model, False), "Gagal bekukan encoder (model tak punya .encoder?)."
n_train = count_parameters(model)
n_total = sum(p.numel() for p in model.parameters())
print(f"Param trainable (decoder+head): {n_train:,} / total {n_total:,} "
      f"({100 * n_train / n_total:.1f}%) -- encoder BEKU")

# Loss TANPA class_weights -- sengaja diisolasi dari sampler: satu2nya mekanisme rebalance
# di run ini cuma sampler class-balanced (cell sebelumnya), bukan ditumpuk dgn reweighting
# loss juga -- biar efeknya jelas ketahuan dari sampler doang, bukan campuran 2 intervensi.
_lc = cfg["training"]["loss"]
loss_fn = make_loss_fn(
    components=[("focal", 0.4), ("tversky", 0.4), ("ce", 0.2)],
    class_weights=None,
    tversky_alpha=_lc.get("tversky_alpha", 0.3),
    tversky_beta=_lc.get("tversky_beta", 0.7),
    focal_gamma=_lc.get("focal_gamma", 2.0),
    device=device,
)
print("Loss: 0.4*Focal + 0.4*Tversky(beta>alpha) + 0.2*CE -- SEMUA unweighted.")
print("Rebalance HANYA lewat sampler class-balanced (cell sebelumnya), bukan loss.")


## Baseline (BEFORE) + tetapkan floor kelas mayoritas (anti-bocor: VAL utk guard, TEST utk laporan)


In [ ]:
# === Baseline (BEFORE) per-kelas IoU: VAL (utk guard) + TEST (utk laporan akhir) ===
# Loop manual baca .npz satu-satu (TANPA DataLoader worker) -> RAM stabil (versi DataLoader
# terbukti bocor puluhan GB di lingkungan ini). VAL menetapkan 'floor' kelas mayoritas;
# TEST HANYA laporan -- JANGAN dipakai utk seleksi checkpoint (cegah kebocoran).
import numpy as np, torch, gc
from forestwatch.training.metrics import compute_confusion_matrix, metric_summary

def _eval_files(mdl, files):
    mdl.eval()
    cm = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    with torch.no_grad():
        for i, fp in enumerate(files):
            d = np.load(fp)
            img = torch.from_numpy(d["img"]).float().unsqueeze(0).to(device)
            pr = mdl(img).argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)
            cm += compute_confusion_matrix(pr, d["lab"], n_classes=N_CLASSES)
            del d, img, pr
            if i % 3000 == 0:
                gc.collect(); torch.cuda.empty_cache()
    return metric_summary(cm, class_names=CLASS_NAMES)

base_val  = _eval_files(model, val_p)
base_test = _eval_files(model, test_p)
base_val_iou  = [r["iou"] for r in base_val["per_class"]]
base_test_iou = [r["iou"] for r in base_test["per_class"]]

EPS = 0.005   # toleransi penurunan kelas mayoritas yg masih diterima (guard)
floor = {c: base_val_iou[c] - EPS for c in STRONG}
print(f"Baseline VAL mIoU={base_val['mean_iou']:.4f} | TEST mIoU={base_test['mean_iou']:.4f} "
      f"| TEST FWIoU={base_test['fwiou']:.4f}")
print("Floor mayoritas (VAL):", {CLASS_NAMES[c]: round(floor[c], 4) for c in STRONG})
print(f"\n{'kelas':<16}{'VAL IoU':>9}{'TEST IoU':>10}")
for c in range(N_CLASSES):
    print(f"{CLASS_NAMES[c]:<16}{base_val_iou[c]:>9.4f}{base_test_iou[c]:>10.4f}")


## Fine-tune (encoder beku, decoder+head) — seleksi checkpoint terjaga + resume-safe


In [ ]:
# === Fine-tune (encoder BEKU) -- seleksi checkpoint TERJAGA (guarded) ===
# Epoch jadi 'best' HANYA bila IoU kelas mayoritas (VAL) >= floor; di antara yg lolos, ambil
# val mIoU tertinggi -> kelas mayoritas dijamin tak turun. Resume-safe (anti sesi Colab mati).
import time, torch
from torch.amp import autocast
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm.auto import tqdm
from forestwatch.training.metrics import set_seed
try:
    from torch.amp import GradScaler; _NEWSCALER = True      # torch >= 2.4
except ImportError:
    from torch.cuda.amp import GradScaler; _NEWSCALER = False  # torch < 2.4 (Jetson)

FT_LR = 1e-4; FT_EPOCHS = 20; FT_PATIENCE = 8
set_seed(cfg["project"]["seed"])
use_amp = torch.cuda.is_available()

trainable = [p for p in model.parameters() if p.requires_grad]   # decoder + head (encoder beku)
optimizer = torch.optim.AdamW(trainable, lr=FT_LR, weight_decay=cfg["training"]["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS)
scaler = (GradScaler(device.type) if _NEWSCALER else GradScaler()) if use_amp else None
iou_pc = MulticlassJaccardIndex(num_classes=N_CLASSES, average=None).to(device)

start_epoch, best_obj, best_epoch, wait, history = 1, -1.0, -1, 0, []
if FT_RESUME.exists():
    try:
        st = torch.load(FT_RESUME, map_location=device)
        model.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"])
        scheduler.load_state_dict(st["scheduler"])
        if scaler and st.get("scaler"): scaler.load_state_dict(st["scaler"])
        start_epoch = st["epoch"] + 1; best_obj = st["best_obj"]; best_epoch = st["best_epoch"]
        wait = st["wait"]; history = st["history"]
        set_encoder_trainable(model, False)   # pastikan encoder tetap beku pasca-resume
        print(f"Resume -> epoch {start_epoch} (best guarded val mIoU={best_obj:.4f})")
    except Exception as e:
        print("Gagal resume:", e)

for ep in range(start_epoch, FT_EPOCHS + 1):
    model.train(); model.encoder.eval()   # encoder BN tetap beku (decoder/head latih)
    tr = 0.0; t0 = time.time()
    for x, y in tqdm(train_loader, desc=f"ep{ep:02d} train", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with autocast(device_type=device.type):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss = loss_fn(model(x), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0); optimizer.step()
        tr += float(loss.item())
    scheduler.step(); tr /= max(len(train_loader), 1)

    model.eval(); iou_pc.reset(); vl = 0.0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"ep{ep:02d} val", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            p = model(x); vl += float(loss_fn(p, y).item()); iou_pc.update(p.argmax(1), y)
    vl /= max(len(val_loader), 1)
    pc = [float(v) for v in iou_pc.compute().tolist()]
    vmiou = sum(pc) / len(pc); weak_miou = sum(pc[c] for c in WEAK) / len(WEAK)
    strong_ok = all(pc[c] >= floor[c] for c in STRONG)
    history.append({"epoch": ep, "train_loss": tr, "val_loss": vl, "val_miou": vmiou,
                    "weak_miou": weak_miou, "val_iou_per_class": [round(v, 4) for v in pc],
                    "strong_ok": bool(strong_ok), "lr": optimizer.param_groups[0]["lr"],
                    "epoch_time_sec": time.time() - t0})
    print(f"ep{ep:02d} | loss {tr:.4f} | val {vl:.4f} | mIoU {vmiou:.4f} | "
          f"weak {weak_miou:.4f} | mayoritas_ok={strong_ok}")
    print("   IoU/kelas:", [round(v, 3) for v in pc])

    if strong_ok and vmiou > best_obj:
        best_obj, best_epoch, wait = vmiou, ep, 0
        torch.save(model.state_dict(), FT_CKPT)
        print(f"   -> best guarded checkpoint (val mIoU={vmiou:.4f}) -> {FT_CKPT.name}")
    else:
        wait += 1
    torch.save({"epoch": ep, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict() if scaler else None,
                "best_obj": best_obj, "best_epoch": best_epoch, "wait": wait,
                "history": history}, FT_RESUME)
    if wait >= FT_PATIENCE:
        print(f"Early stopping di epoch {ep} (patience={FT_PATIENCE})."); break

print(f"\nSelesai. best guarded val mIoU={best_obj:.4f} @ epoch {best_epoch}")
if best_epoch < 0:
    print("PERINGATAN: TAK ada epoch yg lolos guard (semua menurunkan kelas mayoritas).")
    print("-> FT_CKPT tidak tersimpan. Pertahankan baseline; coba turunkan BOOST/FT_LR. JANGAN promosikan.")


In [ ]:
# === Plot kurva fine-tune -> FT_DIR ===
import matplotlib.pyplot as plt
assert history, "history kosong -- jalankan cell fine-tune dulu."
eps = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(eps, [h["train_loss"] for h in history], label="train", lw=2)
ax[0].plot(eps, [h["val_loss"] for h in history], label="val", lw=2)
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(eps, [h["val_miou"] for h in history], color="green", lw=2, label="val mIoU")
ax[1].plot(eps, [h["weak_miou"] for h in history], color="red", lw=2, label="weak mIoU")
ax[1].axhline(base_val["mean_iou"], color="gray", ls="--", label="baseline mIoU")
ax[1].set_title("mIoU (val)"); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].grid(alpha=0.3)
for c in range(N_CLASSES):
    ax[2].plot(eps, [h["val_iou_per_class"][c] for h in history],
               color=CLASS_COLORS[c], lw=1.6, label=CLASS_NAMES[c])
ax[2].set_title("val IoU per-kelas"); ax[2].set_ylim(0, 1)
ax[2].legend(fontsize=7, ncol=2); ax[2].grid(alpha=0.3)
fig.suptitle(MODEL_KEY + " (fine-tune)"); fig.tight_layout()
fig.savefig(FT_DIR / "training_curve_finetune.png", dpi=120, bbox_inches="tight"); plt.show()
print("Disimpan:", FT_DIR / "training_curve_finetune.png")


## Evaluasi akhir di TEST (sekali) — tabel BEFORE/AFTER + vonis jujur


In [ ]:
# === Evaluasi TEST akhir (sekali) + tabel BEFORE/AFTER + vonis jujur ===
import numpy as np, torch
import matplotlib.pyplot as plt
from forestwatch.model.architecture import build_unet, export_to_onnx

if not FT_CKPT.exists():
    print("FT_CKPT tak ada -> fine-tune tak menghasilkan model yg lolos guard.")
    print("VONIS: pertahankan BASELINE. Tidak ada artefak fine-tune utk dipromosikan.")
else:
    ft_model = build_unet(in_channels=cfg["model"]["in_channels"], classes=cfg["model"]["classes"],
                          encoder_weights=cfg["model"]["encoder_weights"], **MODEL_ARCH).to(device)
    ft_model.load_state_dict(torch.load(FT_CKPT, map_location="cpu"))
    ft_test = _eval_files(ft_model, test_p)
    ft_iou = [r["iou"] for r in ft_test["per_class"]]

    print(f"{'kelas':<16}{'BEFORE':>9}{'AFTER':>9}{'delta':>9}")
    drop_majority = []
    for c in range(N_CLASSES):
        dlt = ft_iou[c] - base_test_iou[c]
        flag = ""
        if c in STRONG and dlt < -EPS:
            flag = "  <- MAYORITAS TURUN"; drop_majority.append(c)
        elif c in WEAK and dlt > 0:
            flag = "  <- naik"
        print(f"{CLASS_NAMES[c]:<16}{base_test_iou[c]:>9.4f}{ft_iou[c]:>9.4f}{dlt:>+9.4f}{flag}")
    print(f"\nmIoU  : {base_test['mean_iou']:.4f} -> {ft_test['mean_iou']:.4f} "
          f"({ft_test['mean_iou'] - base_test['mean_iou']:+.4f})")
    print(f"FWIoU : {base_test['fwiou']:.4f} -> {ft_test['fwiou']:.4f} "
          f"({ft_test['fwiou'] - base_test['fwiou']:+.4f})")

    mi_up = ft_test["mean_iou"] >= base_test["mean_iou"]
    if mi_up and not drop_majority:
        print("\nVONIS: BERHASIL -- mIoU naik & kelas mayoritas tak turun. Layak dipromosikan "
              "(lihat cell promosi, set PROMOTE=True).")
    else:
        why = []
        if not mi_up: why.append("mIoU TIDAK naik di test")
        if drop_majority: why.append("mayoritas turun: " + ", ".join(CLASS_NAMES[c] for c in drop_majority))
        print("\nVONIS: BELUM memenuhi target (" + "; ".join(why) + "). Pertahankan baseline / "
              "tune ulang (FT_LR, FT_EPOCHS). JANGAN promosikan.")

    save_json(ft_test, FT_DIR / "metrics_finetune.json")
    save_json({"model_key": MODEL_KEY + "_finetune", **MODEL_ARCH,
               "method": "decoupled cRT (encoder frozen, decoder+head) + class-balanced sampler + unweighted focal_tversky+CE",
               "dataset": BAHAN_DIR.name if BAHAN_DIR else "kaggle_attached_v3",
               "ft_lr": FT_LR, "best_epoch_guarded": best_epoch,
               "baseline_test_miou": base_test["mean_iou"], "finetune_test_miou": ft_test["mean_iou"],
               "baseline_test_per_class_iou": base_test_iou, "finetune_test_per_class_iou": ft_iou,
               "per_class": ft_test["per_class"]}, FT_DIR / "summary_finetune.json")

    cm = np.array(ft_test["confusion_matrix"]); cmn = cm / cm.sum(axis=1, keepdims=True).clip(1)
    fig, axx = plt.subplots(figsize=(8, 6)); im = axx.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            axx.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=9,
                     color="white" if cmn[i, j] > 0.5 else "black")
    axx.set_xticks(range(N_CLASSES)); axx.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    axx.set_yticks(range(N_CLASSES)); axx.set_yticklabels(CLASS_NAMES)
    axx.set_xlabel("Predicted"); axx.set_ylabel("True"); axx.set_title("Confusion - fine-tune")
    fig.colorbar(im, ax=axx); fig.tight_layout()
    fig.savefig(FT_DIR / "confusion_matrix_finetune.png", dpi=120, bbox_inches="tight"); plt.show()

    try:
        export_to_onnx(ft_model, FT_DIR / "model_finetune.onnx",
                       in_channels=cfg["model"]["in_channels"], patch_size=cfg["inference"]["patch_size"])
        print("ONNX:", FT_DIR / "model_finetune.onnx")
    except Exception as e:
        print("ONNX dilewati:", e)
    print("Disimpan:", FT_DIR / "metrics_finetune.json", "| summary_finetune.json | confusion_matrix_finetune.png")


In [ ]:
# === (Opsional) Promosikan model fine-tune ke lokasi baseline -- HANYA bila VONIS berhasil ===
# Default OFF. Set PROMOTE=True secara SADAR setelah cek tabel BEFORE/AFTER. Baseline di-backup
# dulu (best_model_prefinetune_backup.pt) -> tetap reversible. Di Kaggle: tak ada mount Drive,
# jadi paket semua artefak FT_DIR jadi 1 zip (selalu, lepas dari PROMOTE) supaya tinggal
# download via panel Output Kaggle -> upload manual ke Drive di path DRIVE_TARGET_NAME.
PROMOTE = False
import shutil

if ENV not in ("colab", "lab"):
    zip_base = FT_DIR.parent / (FT_DIR.name + "_package")
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=FT_DIR)
    print("Paket siap diunduh:", zip_path)
    print("Langkah selanjutnya (manual, Kaggle tak ada mount Drive):")
    print("  1. Kaggle -> Save Version -> tunggu selesai -> buka tab 'Output' notebook ini.")
    print(f"  2. Unduh '{Path(zip_path).name}', lalu di Google Drive buat folder:")
    print(f"     Satria Data 3.0/{DRIVE_TARGET_NAME}")
    print("  3. Ekstrak isi zip ke folder itu.")
    print("  4. (Opsional) Kalau mau promosikan jadi baseline baru, jalankan ulang cell ini")
    print("     dgn ENV='colab'/'lab' (Drive ter-mount) setelah file ada di Drive.")
elif not PROMOTE:
    print("PROMOTE=False -- tidak menyalin apa pun. Baseline tetap aktif.")
    print("Hasil fine-tune tetap tersimpan lengkap di:", FT_DIR)
elif not FT_CKPT.exists():
    print("FT_CKPT tak ada -- tak ada yg dipromosikan.")
else:
    bak = BASE_DIR / "best_model_prefinetune_backup.pt"
    if not bak.exists():
        shutil.copy(BASE_DIR / "best_model.pt", bak); print("Backup baseline ->", bak)
    shutil.copy(FT_CKPT, BASE_DIR / "best_model.pt")
    shutil.copy(FT_DIR / "metrics_finetune.json", BASE_DIR / "metrics.json")
    if (FT_DIR / "model_finetune.onnx").exists():
        shutil.copy(FT_DIR / "model_finetune.onnx", BASE_DIR / "model.onnx")
    print("Dipromosikan ->", BASE_DIR, "(baseline sudah di-backup; reversible).")
